# Reto Spaceship Titanic - Avance 1

| | |
|---|---|
| **Equipo** | Israel González Huerta A01751433<br>Luis David Pozos Tamez A01800657<br>Adrián Proaño Bernal A01752615<br>Juan Pablo Solís Gómez A01800430<br>Karol Alexis Alvarado Davila A01751711 |
| **Fecha** | 31 de agosto de 2026 |

---

## 1. Carga y comprensión del dataset

### 1.1 Carga de `train.csv` y `test.csv`

Carga de los conjuntos de entrenamiento y prueba provistos por Kaggle. No se realiza split adicional: todo el trabajo de preparación se desarrolla sobre `train.csv`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler


In [3]:
train = pd.read_csv('../data/titanic_data/train.csv')
test = pd.read_csv('../data/titanic_data/test.csv')

### 1.2 Inspección de estructura: shape, tipos de variables y primeras filas

Revisión del número de filas y columnas, tipos de datos (`dtypes`) por variable, y muestra de las primeras observaciones (`head`) para comprender el formato general del dataset.

In [4]:
print(test.shape)
print(train.shape)

(4277, 13)
(8693, 14)


In [5]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [6]:
test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [8]:
train.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

### 1.3 Identificación de la variable target y tipo de problema

`Transported` es nuestra variable objetivo. El problema que se está intentando resolver en este escenario es identificar a los pasajeros que fueron transportados a la dimensión alterna, por lo que cuadra perfectamente con el dato booleano que aparece en el training set. 

El tipo de problema al que nos enfrentamos es una **clasificación binaria supervisada**, ya que la predicción solo puede caer en dos categorías, y además tenemos acceso a una muestra con valores predefinidos de True/False para `Transported` (test.csv). A partir de esto podemos verificar algunas de las respuestas que da el modelo y proporcionarle retroalimentación.

## 2. Análisis exploratorio (EDA)

### 2.1 Valores faltantes: porcentaje por columna y clasificación (informativo vs. real)

Cálculo del porcentaje de faltantes por columna. Cada variable se clasifica como **faltante informativo** o **faltante real**, justificando cada clasificación con el contexto del dataset descrito en Kaggle.

In [10]:
faltantes = train.isnull().sum()
faltantes / len(train) * 100

PassengerId     0.000000
HomePlanet      2.312205
CryoSleep       2.496261
Cabin           2.289198
Destination     2.093639
Age             2.059128
VIP             2.335212
RoomService     2.082135
FoodCourt       2.105142
ShoppingMall    2.392730
Spa             2.105142
VRDeck          2.162660
Name            2.300702
Transported     0.000000
dtype: float64

### 2.2 Variables numéricas: distribución, skewness y outliers

Histogramas de las variables numéricas, cálculo de asimetría (skewness) y detección de outliers mediante el método de Tukey/rango intercuartílico (IQR).

### 2.3 Variables categóricas: cardinalidad, distribución y desbalance

Análisis de la cardinalidad de cada variable categórica, distribución de sus categorías y detección de variables con desbalance extremo entre clases.

### 2.4 Relación de cada variable con el target

Crosstabs entre variables categóricas y `Transported`, y boxplots de variables numéricas segmentados por `Transported`. Este análisis es la base para justificar qué variables entran y cuáles se descartan en la siguiente etapa.

## 3. Selección y descarte de variables

### 3.1 Variables incluidas en el pipeline

Listado de variables que se conservan para el modelado, con la evidencia del EDA que respalda cada inclusión (relación con el target, faltantes manejables, cardinalidad razonable, etc.).

### 3.2 Variables descartadas

Listado de variables excluidas del pipeline, justificando cada descarte con evidencia concreta: porcentaje de faltantes, desbalance extremo, ausencia de relación con el target, cardinalidad excesiva o redundancia con otra variable.

### 3.3 Evaluación de binarización para variables con desbalance extremo

Antes de descartar variables con clases muy desbalanceadas (por ejemplo, servicios de gasto a bordo), se evalúa si transformarlas en indicadores binarios (`HasPool`, `HasCabin`, etc.) preserva señal útil para el target.

## 4. Manejo de faltantes

### 4.1 Imputación de faltantes informativos

Estrategia de imputación aplicada a las variables cuyo faltante fue clasificado como informativo en el paso 2.1 (por ejemplo, imputación condicionada al valor de otra variable relacionada), con su justificación.

### 4.2 Imputación de faltantes reales

Estrategia de imputación aplicada a las variables con faltantes reales (por ejemplo, medidas de tendencia central para numéricas o moda/categoría "Unknown" para categóricas), con su justificación.

### 4.3 Resumen de estrategia de imputación por variable

Tabla o resumen que consolida, por variable, el tipo de faltante y la estrategia de imputación elegida, evitando aplicar un único criterio genérico sin argumento.

## 5. Codificación de variables categóricas

### 5.1 Codificación de variables nominales

Aplicación de codificación a variables categóricas sin orden natural, justificando la elección según cardinalidad y naturaleza de cada variable.

### 5.2 Codificación de variables ordinales

Evaluación de si alguna variable categórica presenta orden natural y, de ser así, aplicación de una codificación ordinal justificada.

### 5.3 Verificación de ausencia de columnas tipo `object`

Comprobación de que, tras la codificación, ninguna columna del dataset conserva el tipo `object`.

## 6. Transformación de variables numéricas

### 6.1 Evaluación del sesgo (skewness)

Revisión del coeficiente de asimetría de cada variable numérica para identificar cuáles presentan sesgo significativo.

### 6.2 Aplicación de transformaciones justificadas

Aplicación de una transformación (p. ej. logarítmica o `PowerTransformer`) a las variables con sesgo relevante, justificando la elección según el tipo y grado de sesgo detectado.

### 6.3 Comparación de distribuciones antes y después

Visualización comparativa (histogramas) de cada variable transformada, mostrando su distribución original frente a la distribución tras la transformación.

## 7. Escalado

### 7.1 Selección del escalador según presencia de outliers

Decisión entre `StandardScaler` y `RobustScaler` (u otra alternativa) para cada grupo de variables numéricas, en función de la presencia de outliers detectada en el paso 2.2.

### 7.2 Aplicación del escalado

Ajuste y aplicación del escalador elegido sobre las variables numéricas del dataset, con justificación de la elección.

## 8. Verificación final

### 8.1 Confirmación de ausencia de nulos

Comprobación de que el conjunto de datos procesado no contiene valores nulos.

### 8.2 Confirmación de ausencia de columnas tipo `object`

Comprobación de que todas las columnas del dataset final son de tipo numérico.

### 8.3 Shape final del dataset procesado

Reporte de la dimensión final (filas x columnas) del dataset tras todas las transformaciones.

### 8.4 Tabla resumen de decisiones

Tabla que consolida, para cada variable original: tipo de variable, tratamiento aplicado (imputación, codificación, transformación, escalado o descarte) y justificación de la decisión.

| Variable | Tipo | Tratamiento | Justificacion |
|---|---|---|---|
| PassengerId | | | |
| HomePlanet | | | |
| CryoSleep | | | |
| Cabin | | | |
| Destination | | | |
| Age | | | |
| VIP | | | |
| RoomService | | | |
| FoodCourt | | | |
| ShoppingMall | | | |
| Spa | | | |
| VRDeck | | | |
| Name | | | |
| Transported | | | |

### 8.5 Conclusiones y hallazgos

Síntesis de los principales hallazgos del análisis exploratorio y de las decisiones de preparación de datos, y su relevancia para el modelado posterior.